In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
#!pip install langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH03-OutputParser")

In [5]:
# 실시간 출력을 위한 import
from langchain_teddynote.messages import stream_response

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


llm = ChatOpenAI(temperature=0, model_name="gpt-4.1-mini")

In [ ]:
##1. PydanticOutputParser를 통해 출력 파서 추가 및 구조화된 답변 받기
from langchain_teddynote.messages import stream_response
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-OutputPraser")

llm = ChatOpenAI(temperature=0, model_name="gpt-4.1-mini")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputPraser


In [12]:
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

In [13]:
from itertools import chain
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "다음의 이메일 내용 중 중요한 내용을 추출해 주세요.\n\n{email_conversation}"
)

llm = ChatOpenAI(temperature=0, model_name="gpt-4.1-mini")

chain = prompt | llm

answer = chain.stream({"email_conversation": email_conversation})

output = stream_response(answer, return_output=True)

중요 내용 요약:

- 김철수 상무(바이크코퍼레이션)가 이은채 대리(테디인터내셔널)에 "ZENESIS" 자전거 유통 협력 제안
- ZENESIS 모델의 상세 브로슈어 요청 (기술 사양, 배터리 성능, 디자인 정보 포함)
- 유통 전략 및 마케팅 계획 구체화를 위해 정보 필요
- 1월 15일 화요일 오전 10시에 미팅 제안 (귀사 사무실에서)

In [15]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

In [16]:
parser = PydanticOutputParser(pydantic_object=EmailSummary)

In [17]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


In [18]:
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT
{format}
"""
)

In [19]:
prompt = prompt.partial(format=parser.get_format_instructions())
prompt

PromptTemplate(input_variables=['email_conversation', 'question'], input_types={}, partial_variables={'format': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required

In [20]:
#프롬프트와 LLM 체인 생성
chain = prompt | llm

response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 중요 내용을 추출해 주세요.",
    }
)

output = stream_response(response, return_output=True)

```json
{
  "person": "김철수",
  "email": "chulsoo.kim@bikecorporation.me",
  "subject": "\"ZENESIS\" 자전거 유통 협력 및 미팅 일정 제안",
  "summary": "바이크코퍼레이션 김철수 상무가 ZENESIS 자전거의 상세 브로슈어(기술 사양, 배터리 성능, 디자인) 요청과 유통 및 마케팅 협력 논의를 위해 미팅을 제안함.",
  "date": "1월 15일 오전 10시"
}
```

In [21]:
#문자열 타입을 객체 타입으로 변환
structured_output = parser.parse(output)
print(structured_output)

person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary='바이크코퍼레이션 김철수 상무가 ZENESIS 자전거의 상세 브로슈어(기술 사양, 배터리 성능, 디자인) 요청과 유통 및 마케팅 협력 논의를 위해 미팅을 제안함.' date='1월 15일 오전 10시'


In [22]:
structured_output.person

'김철수'

In [23]:
chain = prompt | llm | parser

response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해 주세요.",
    }
)

response

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='바이크코퍼레이션 김철수 상무가 ZENESIS 자전거에 대한 상세 브로슈어(기술 사양, 배터리 성능, 디자인)를 요청하고, 유통 전략과 마케팅 계획 수립을 위해 협력 가능성을 논의하고자 1월 15일 오전 10시에 미팅을 제안함.', date='1월 15일 오전 10시')

In [ ]:
##2. with_structured_output() 바인딩
#해당 답변은 구조화된 답변이 아님
llm = ChatOpenAI(
    temperature=0, model_name="gpt-4.1-mini"
)

llm.invoke("대한민국의 수도는 뭐야?")

AIMessage(content='대한민국의 수도는 서울입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 15, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_7725bead7d', 'id': 'chatcmpl-EOYqjZlWu1FXJH99XAt4Wp0vGte56', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0a7da-fa40-7681-b84e-3743354c6f28-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 8, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [25]:
llm_with_structured = ChatOpenAI(
    temperature=0, model_name="gpt-4.1-mini"
).with_structured_output(EmailSummary)

In [26]:
answer = llm_with_structured.invoke(email_conversation)
answer.person

'김철수'

In [27]:
##4. 쉼표로 구분된 리스트 출력 파서
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 콤마로 구분된 리스트 출력 파서 초기화
output_parser = CommaSeparatedListOutputParser()

# 출력 형식 지침 가져오기
format_instructions = output_parser.get_format_instructions()
# 프롬프트 템플릿 설정
prompt = PromptTemplate(
    # 주제에 대한 다섯 가지를 나열하라는 템플릿
    template="List five {subject}.\n{format_instructions}",
    input_variables=["subject"],  # 입력 변수로 'subject' 사용
    # 부분 변수로 형식 지침 사용
    partial_variables={"format_instructions": format_instructions},
)

# ChatOpenAI 모델 초기화
model = ChatOpenAI(temperature=0)

# 프롬프트, 모델, 출력 파서를 연결하여 체인 생성
chain = prompt | model | output_parser

In [28]:
chain.invoke({"subject": "대한민국 관광명소"})

['경복궁', '남산타워', '부산 해운대해수욕장', '제주도 성산일출봉', '경주 불국사temples']

In [29]:
for s in chain.stream({"subject": "대한민국 관광명소"}):
    print(s)

['경복궁']
['남산타워']
['부산 해운대해수욕장']
['제주도 성산일출봉']
['경주 불국사temples']


In [ ]:
##5.구조화된 출력파서
#llm에 대한 답변을 딕셔너리 형식으로 정의. gpt/claude 보다 인텔리전스가 낮은 로컬 모델에 더 적합
#langchain_classic으로 해야 돌아감 패키지 변경
from langchain_teddynote import logging
logging.langsmith("CH03-OutputParser")

from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

response_schemas = [
    ResponseSchema(
        name="answer",
        description="사용자 질문에 대한 답변"
    ),
    ResponseSchema(
        name="source",
        description="사용자의 질문에 답하기 위해 사용된 '출처', '웹사이트 주소'이어야 합니다"
    ),
]

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser


In [38]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [39]:
print(output_parser.get_format_instructions)

<bound method StructuredOutputParser.get_format_instructions of StructuredOutputParser(response_schemas=[ResponseSchema(name='answer', description='사용자 질문에 대한 답변', type='string'), ResponseSchema(name='source', description="사용자의 질문에 답하기 위해 사용된 '출처', '웹사이트 주소'이어야 합니다", type='string')])>


In [41]:
format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    template="answer the users question as best as possible.\n{format_instructions}\n{question}",
    # 입력 변수로 'question'을 사용합니다.
    input_variables=["question"],
    # 부분 변수로 'format_instructions'을 사용합니다.
    partial_variables={"format_instructions": format_instructions},
)

In [42]:
model = ChatOpenAI(temperature=0)  
chain = prompt | model | output_parser  

In [ ]:
#답변이 딕셔너리 형식으로 표시
chain.invoke({"question": "대한민국의 수도는 어디인가요?"})

{'answer': '서울', 'source': 'https://ko.wikipedia.org/wiki/%EC%84%9C%EC%9A%B8'}

In [46]:
##6.Json 형식 출력 파서
#모델 용량이 충분히 커야 원하는 Json 형식으로 생성 가능함

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-OutputParser")

model = ChatOpenAI(temperature=0, model_name="gpt-4.1-mini")


LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser


In [48]:
class Topic(BaseModel):
    description: str = Field(description="주제에 대한 간결한 설명")
    hashtags: str = Field(description="해시태그 형식의 키워드(2개 이상)")

In [49]:
question = "지구 온난화의 심각성에 대해 알려주세요"

parser = JsonOutputParser(pydantic_object=Topic)
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [51]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("user", "#format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser

answer = chain.invoke({"question": question})

In [52]:
answer["description"]

'지구 온난화는 지구 평균 기온 상승으로 인해 극심한 기후 변화, 해수면 상승, 생태계 파괴 등 심각한 환경 문제를 초래합니다.'

In [53]:
answer["hashtags"]

'#지구온난화 #기후변화 #환경보호 #지구살리기'

In [58]:
##7.Pandas 데이터프레임 출력 파서
#langchain_classic으로 해야함 
import pprint
from typing import Any, Dict

import pandas as pd
from langchain_classic.output_parsers import PandasDataFrameOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-OutputParser")


LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser


In [59]:
# ChatOpenAI 모델 초기화 (gpt-3.5-turbo 모델 사용을 권장)
model = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")

In [ ]:
#출력 형식을 깔끔하게 하는 용도
def format_parser_output(parser_output: Dict[str, Any]) -> None:
    for key in parser_output.keys():
        parser_output[key] = parser_output[key].to_dict()
    return pprint.PrettyPrinter(width=4, compact=True).pprint(parser_output)

In [63]:
df = pd.read_csv("./titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [64]:
parser = PandasDataFrameOutputParser(dataframe=df)

print(parser.get_format_instructions())

The output should be formatted as a string as the operation, followed by a colon, followed by the column or row to be queried on, followed by optional array parameters.
1. The column names are limited to the possible columns below.
2. Arrays must either be a comma-separated list of numbers formatted as [1,3,5], or it must be in range of numbers formatted as [0..4].
3. Remember that arrays are optional and not necessarily required.
4. If the column is not in the possible columns or the operation is not a valid Pandas DataFrame operation, return why it is invalid as a sentence starting with either "Invalid column" or "Invalid operation".

As an example, for the formats:
1. String "column:num_legs" is a well-formatted instance which gets the column num_legs, where num_legs is a possible column.
2. String "row:1" is a well-formatted instance which gets row 1.
3. String "column:num_legs[1,2]" is a well-formatted instance which gets the column num_legs for rows 1 and 2, where num_legs is a p

In [67]:
#parser 이용 특정 열의 값 조회
df_query = "Age column을 조회해 주세요."

prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{question}\n",
    input_variables=["question"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)

chain = prompt | model | parser

parser_output = chain.invoke({"question": df_query})

format_parser_output(parser_output)

{'Age': {0: 22.0,
         1: 38.0,
         2: 26.0,
         3: 35.0,
         4: 35.0,
         5: nan,
         6: 54.0,
         7: 2.0,
         8: 27.0,
         9: 14.0,
         10: 4.0,
         11: 58.0,
         12: 20.0,
         13: 39.0,
         14: 14.0,
         15: 55.0,
         16: 2.0,
         17: nan,
         18: 31.0,
         19: nan}}


In [68]:
df_query = "Retrieve the first row."
parser_output = chain.invoke({"question": df_query})
format_parser_output(parser_output)

{'0': {'Age': 22.0,
       'Cabin': nan,
       'Embarked': 'S',
       'Fare': 7.25,
       'Name': 'Braund, '
               'Mr. '
               'Owen '
               'Harris',
       'Parch': 0,
       'PassengerId': 1,
       'Pclass': 3,
       'Sex': 'male',
       'SibSp': 1,
       'Survived': 0,
       'Ticket': 'A/5 '
                 '21171'}}


In [69]:
df["Age"].head().mean()

np.float64(31.2)

In [70]:
df_query = "Retrieve the average of the Ages from row 0 to 4."
parser_output = chain.invoke({"question": df_query})
print(parser_output)

{'mean': np.float64(31.2)}


In [71]:
df_query = "Calculate average 'Fare' rate."
parser_output = chain.invoke({"question": df_query})
print(parser_output)

{'mean': np.float64(22.19937)}


In [73]:
df["Fare"].mean()

np.float64(22.19937)

In [77]:
##8.날짜 형식 출력 파서
#langchain_classic으로 해야함
from langchain_classic.output_parsers import DatetimeOutputParser
from langchain_classic.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

load_dotenv()
logging.langsmith("CH03-OutputParser")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser


In [78]:
output_parser = DatetimeOutputParser()
output_parser.format = "%Y-%m-%d"

In [79]:
print(output_parser.get_format_instructions())

Write a datetime string that matches the following pattern: '%Y-%m-%d'.

Examples: 2026-09-16, 2025-09-16, 2026-09-15

Return ONLY this string, no other words!


In [80]:
template = """Answer the users question:
#Format Instructions:
{format_instructions}

#Question:
{question}

#Answer:"""

prompt = PromptTemplate.from_template(
    template,
    partial_variables={
        "format_instructions": output_parser.get_format_instructions()
    },
)

prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={'format_instructions': "Write a datetime string that matches the following pattern: '%Y-%m-%d'.\n\nExamples: 2026-09-16, 2025-09-16, 2026-09-15\n\nReturn ONLY this string, no other words!"}, template='Answer the users question:\n#Format Instructions:\n{format_instructions}\n\n#Question:\n{question}\n\n#Answer:')

In [81]:
chain = prompt | ChatOpenAI() | output_parser
output = chain.invoke({"question": "Google 이 창업한 연도"})


In [82]:
output.strftime("%Y-%m-%d")

'1998-09-04'

In [83]:
##9.열거형 출력 파서
#classic 사용해야함
from enum import Enum
from langchain_classic.output_parsers.enum import EnumOutputParser
from langchain_classic.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-OutputParser")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser


In [84]:
class Colors(Enum):
    RED = "빨간색"
    GREEN = "초록색"
    BLUE = "파란색"

In [85]:
Colors.RED

<Colors.RED: '빨간색'>